# Receipt → JSON: baseline eval → firectl SFT (end-to-end)

A full managed fine-tuning walkthrough for a vision task (receipt image → structured JSON), using **firectl** (Fireworks' CLI) for deployment + training and **Eval Protocol** for measurement:

1. **Download data** — CORD-v2 receipts → multimodal SFT JSONL (train + holdout).
2. **Get/create a deployment** of the base model (needed to run inference for the baseline).
3. **Eval Protocol baseline** on the holdout set — measure the base model *before* SFT.
4. **firectl** — upload the dataset and launch the supervised fine-tuning job.
5. (after training) deploy the tuned model and re-run the same eval to see the lift.

> Step 2 comes before step 3 on purpose: the base VLM isn't serverless on every account, so we stand up a dedicated deployment to serve it for the baseline. Both the deployment and the fine-tune cost GPU time on your account.

**Prereqs:** Jupyter kernel = conda `cookbook` env, `FIREWORKS_API_KEY` in `training/.env`, and your Fireworks `ACCOUNT_ID`.

In [1]:
# Run once if needed. Installs eval-protocol + datasets, and firectl (Fireworks CLI).
import sys
!{sys.executable} -m pip install -q -e "../../../.[eval]" datasets
# firectl: https://docs.fireworks.ai/tools-sdks/firectl/firectl  (macOS shown; see docs for Linux)
!command -v firectl >/dev/null 2>&1 && echo "firectl present: $(firectl version 2>/dev/null | head -1)" || echo "Install firectl: brew install fw-ai/firectl/firectl  (or see docs)"

Install firectl: brew install fw-ai/firectl/firectl  (or see docs)


In [ ]:
# --- edit these ---
ACCOUNT_ID = ""   # <-- your Fireworks account id (required for resource names)
BASE_MODEL = "accounts/fireworks/models/qwen3-vl-8b-instruct"

DATASET_ID = "cord-receipts"          # firectl dataset id (train split)
OUTPUT_MODEL_ID = "sft-cord-qwen3vl"  # fine-tuned model id firectl will create

TRAIN_MAX = 400     # CORD train is 800; cap for a cheaper run
EVAL_MAX = 50       # holdout receipts to score
EPOCHS = 3
LORA_RANK = 16
CONCURRENCY = 4

# Inference model string for eval (litellm `fireworks_ai/...`). For a dedicated
# deployment of a base model, address it as <model>#<deployment_id>. Filled in by
# the deployment cell below; override manually if you already have a deployment.
INFERENCE_MODEL = None
TUNED_INFERENCE_MODEL = None  # set after training + deploying the tuned model

In [ ]:
import asyncio
import json
import os
import subprocess
import sys
from pathlib import Path

import litellm
from dotenv import load_dotenv
from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import SingleTurnRolloutProcessor
from eval_protocol.pytest.types import RolloutProcessorConfig

HERE = Path.cwd()
training_dir = next(
    (p for p in [HERE, *HERE.parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../../").resolve(),
)
load_dotenv(training_dir / ".env")
if not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")
if not ACCOUNT_ID:
    raise ValueError("Set ACCOUNT_ID in the config cell.")
litellm.drop_params = True


def sh(cmd: str):
    """Run a shell command, stream output, raise on failure."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.stdout

## 1. Download data → SFT JSONL (train + holdout)

`prepare_cord_sft.py` downloads CORD-v2, flattens each receipt's `gt_parse` into a clean schema, base64-encodes the image, and writes OpenAI multimodal chat rows. We make a **train** split (for firectl SFT) and a held-out **test** split (for the Eval Protocol baseline).

In [ ]:
def prep(split, max_examples, out):
    if Path(out).exists():
        print(f"{out} exists, skipping")
    else:
        subprocess.run([sys.executable, "prepare_cord_sft.py", "--split", split,
                        "--max-examples", str(max_examples), "--out", out], check=True)
    return out

TRAIN_JSONL = prep("train", TRAIN_MAX, "cord_train.jsonl")
HOLDOUT_JSONL = prep("test", EVAL_MAX, "cord_test.jsonl")
holdout = [json.loads(l) for l in open(HOLDOUT_JSONL)]
print(f"train={sum(1 for _ in open(TRAIN_JSONL))}  holdout={len(holdout)}")

## 2. Get or create a deployment of the base model

We need the base VLM served to run the baseline. List deployments; if none serves `BASE_MODEL`, create one. (On-demand deployment — remember to delete it when done.)

In [ ]:
# Confirm the base model is tunable, then get-or-create a deployment to serve it.
sh(f"firectl model get -a fireworks {BASE_MODEL.split('/')[-1]} | grep -i -E 'tunable|supports lora' || true")

existing = sh("firectl list deployments -o json 2>/dev/null || firectl deployment list -o json")
dep_id = None
try:
    for d in json.loads(existing):
        if d.get("baseModel") == BASE_MODEL and d.get("state") in ("READY", "DEPLOYING", "STATE_READY"):
            dep_id = d["name"].split("/")[-1]
            break
except Exception:
    pass

if dep_id:
    print("Reusing deployment:", dep_id)
else:
    out = sh(f"firectl deployment create {BASE_MODEL} --wait")
    # parse the created deployment id from output (e.g. 'accounts/<acct>/deployments/<id>')
    import re
    m = re.search(r"deployments/([a-z0-9-]+)", out)
    dep_id = m.group(1) if m else None
    print("Created deployment:", dep_id)

INFERENCE_MODEL = f"fireworks_ai/{BASE_MODEL}#accounts/{ACCOUNT_ID}/deployments/{dep_id}"
print("INFERENCE_MODEL =", INFERENCE_MODEL)

## 3. Eval Protocol baseline on the holdout

Each holdout receipt becomes an `EvaluationRow` (image+instruction as the user turn, gold JSON as `ground_truth`). `SingleTurnRolloutProcessor` runs the deployed base model; we grade with a field-level scorer (scalar exact-match + line-item F1).

In [ ]:
SCALARS = ["subtotal", "tax", "service", "total"]


def parse_json(text):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        i, j = text.find("{"), text.rfind("}")
        try:
            return json.loads(text[i : j + 1])
        except Exception:
            return None


def _n(v):
    return str(v).strip().lower() if v is not None else None


def _items(o):
    return {(_n(it.get("name")), _n(it.get("price"))) for it in (o or {}).get("items", []) or [] if isinstance(it, dict)}


def field_score(pred, gold):
    if not isinstance(pred, dict):
        return 0.0
    parts = [1.0 if _n(pred.get(k)) == _n(gold.get(k)) else 0.0 for k in SCALARS]
    p, g = _items(pred), _items(gold)
    if p or g:
        tp = len(p & g)
        prec = tp / len(p) if p else 0.0
        rec = tp / len(g) if g else 0.0
        parts.append(2 * prec * rec / (prec + rec) if (prec + rec) else 0.0)
    return sum(parts) / len(parts)


def grade(row: EvaluationRow) -> EvaluateResult:
    pred = parse_json(str(row.messages[-1].content))
    gold = json.loads(row.ground_truth)
    return EvaluateResult(score=field_score(pred, gold), reason="ok" if pred else "unparseable")


def build_rows(samples):
    rows = []
    for i, s in enumerate(samples):
        rows.append(EvaluationRow(messages=[s["messages"][0]], ground_truth=s["messages"][1]["content"]))
        rows[-1].input_metadata.row_id = f"receipt-{i}"
    return rows


async def eval_protocol_score(inference_model, samples):
    processor = SingleTurnRolloutProcessor(drop_trailing_assistant_messages=True)
    config = RolloutProcessorConfig(
        completion_params={"model": inference_model, "temperature": 0.0, "max_tokens": 2048},
        mcp_config_path="",
        semaphore=asyncio.Semaphore(CONCURRENCY),
    )
    results = await asyncio.gather(*processor(build_rows(samples), config), return_exceptions=True)
    rows = [r for r in results if isinstance(r, EvaluationRow)]
    errs = len(results) - len(rows)
    scores = [grade(r).score for r in rows]
    return (sum(scores) / len(scores) if scores else 0.0), errs


base_score, base_errs = await eval_protocol_score(INFERENCE_MODEL, holdout)
print(f"BASE field_score = {base_score:.1%} over {len(holdout)} receipts ({base_errs} errors)")

## 4. firectl: upload dataset + launch SFT

`firectl dataset create` uploads the train JSONL; `firectl sftj create` launches a LoRA supervised fine-tuning job against `BASE_MODEL`.

In [ ]:
# Upload the training dataset (idempotent-ish: ignore error if it already exists).
sh(f"firectl dataset create {DATASET_ID} {Path(TRAIN_JSONL).resolve()} || echo '(dataset may already exist)'")

# Launch the supervised fine-tuning job.
out = sh(
    f"firectl sftj create "
    f"--base-model {BASE_MODEL} "
    f"--dataset {DATASET_ID} "
    f"--output-model {OUTPUT_MODEL_ID} "
    f"--epochs {EPOCHS} "
    f"--lora-rank {LORA_RANK}"
)
import re
m = re.search(r"supervisedFineTuningJobs/([a-z0-9-]+)", out)
JOB_ID = m.group(1) if m else None
print("SFT job:", JOB_ID)

In [ ]:
# Poll job status (re-run this cell until COMPLETED).
sh(f"firectl sftj get {JOB_ID} | grep -i -E 'state|status|output model' || firectl sftj get {JOB_ID}")

## 5. Deploy the tuned model and re-measure

Once the job is `COMPLETED`, deploy the resulting model and re-run the exact same Eval Protocol scorer on the holdout.

In [ ]:
TUNED_MODEL = f"accounts/{ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}"
out = sh(f"firectl deployment create {TUNED_MODEL} --wait")
import re
m = re.search(r"deployments/([a-z0-9-]+)", out)
tuned_dep = m.group(1) if m else None
TUNED_INFERENCE_MODEL = f"fireworks_ai/{TUNED_MODEL}#accounts/{ACCOUNT_ID}/deployments/{tuned_dep}"

tuned_score, tuned_errs = await eval_protocol_score(TUNED_INFERENCE_MODEL, holdout)
print(f"BASE  field_score = {base_score:.1%}")
print(f"TUNED field_score = {tuned_score:.1%}  ({tuned_errs} errors)")
print(f"\nLift: {tuned_score - base_score:+.1%}")

## Cleanup

On-demand deployments bill while up. Delete them when done:
```bash
firectl deployment delete <BASE_DEPLOYMENT_ID>
firectl deployment delete <TUNED_DEPLOYMENT_ID>
```

**Next:** distillation (teacher VLM labels a larger unlabeled receipt pool) and DPO (prefer abstaining on illegible fields) build on the tuned model from here.